# Debug Scraper Surya Malang

Local HTML first, then optional live list/article checks. Output list CSV: `csv/surya_malang_list_debug.csv`.

In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.common import cutoff_date, parse_date, records_to_df
import scrapers.suryamalang as scraper
scraper = importlib.reload(scraper)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 160)

SAMPLE_PATH = PROJECT_ROOT / 'html_samples/suryamalang.html'
OUTPUT_PATH = PROJECT_ROOT / 'csv/surya_malang_list_debug.csv'
MAX_PAGES = 200
TITLE_LIMIT = 90
cutoff = cutoff_date()

print('project:', PROJECT_ROOT)
print('module:', scraper.__file__)
print('sample:', SAMPLE_PATH)
print('cutoff:', cutoff.date())


In [ ]:

def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df
    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


In [ ]:

def parse_local_list(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = []
    for card in soup.select('ul.lsi > li'):
        title_tag = card.select_one('h3 a[href]')
        date_tag = card.select_one('div.grey span.grey, div.grey')
        excerpt_tag = card.select_one('h4.grey2')
        image_tag = card.select_one('img')
        if not title_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': scraper.BASE_URL,
            'title': title_tag.get_text(' ', strip=True),
            'url': urljoin(scraper.BASE_URL, title_tag['href']),
            'published_date': scraper.clean_date(date_tag.get_text(' ', strip=True) if date_tag else None),
            'excerpt': excerpt_tag.get_text(' ', strip=True) if excerpt_tag else None,
            'image_url': image_tag.get('src') if image_tag else None,
        })
    return records_to_df(rows)


## Local HTML list parse

In [ ]:
html_text = SAMPLE_PATH.read_text(errors='replace')
local_df = parse_local_list(html_text)
local_df = ensure_debug_columns(local_df)
print_debug_rows(local_df)
local_df = audit_list(local_df)


## Live list scrape

In [ ]:
live_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
live_df = ensure_debug_columns(live_df)
print_debug_rows(live_df)
live_df = audit_list(live_df)


## Save debug list CSV

In [ ]:
df_to_save = live_df if 'live_df' in globals() and len(live_df) else local_df
OUTPUT_PATH.parent.mkdir(exist_ok=True)
df_to_save.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)


## Optional article sample check

In [ ]:
sample_rows = (live_df if 'live_df' in globals() and len(live_df) else local_df).head(5)
article_rows = []
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")
article_df = pd.DataFrame(article_rows)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
